# Make Figure 1 of the paper

In [ ]:
import os
import sys
import pandas as pd
import pygmt
from obspy import UTCDateTime
import tempfile
from pyproj import Transformer
from shapely.geometry import Point
from shapely.ops import unary_union, transform
import geopandas as gpd


from obspy.clients.fdsn import Client

notebook_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(notebook_dir, '../utils/'))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
import fitz  # PyMuPDF


import matplotlib.dates as mdates


from qc_utils import calc_active_days_of_stas


In [ ]:
# pip install pymupdf

## Make figure 1(a), the station map

In [ ]:
# Define the client 
client_inventory = Client("IRIS")


In [ ]:
# Load the station data
stations = pd.read_csv('../data/datasets_all_regions/all_stations_2010_2015_ver3.csv',index_col=0)
stations

In [ ]:
# Obtain the inventory for all stations
bulk_sta = []
for index, sta in stations.iterrows():
    network = sta['network']
    station = sta['station']
    loc = '*'
    ch = '?H?' 
    t1 = UTCDateTime('2010-01-01T00:00:00')
    t2 = UTCDateTime('2015-12-31T23:59:59')
    bulk_sta.append([network,station,loc,ch,t1,t2])
    
inv = client_inventory.get_stations_bulk(bulk_sta)

In [ ]:
# Calculate the active days of each station
start_time = UTCDateTime('2010-01-01T00:00:00')
end_time = UTCDateTime('2015-12-31T23:59:59')
used_stations = calc_active_days_of_stas(inv, start_time, end_time)

# Combine data into a single DataFrame
stations = pd.DataFrame(used_stations)

In [ ]:
stations

In [ ]:
# Extract the start dates of the stations
sta_start_dates = []
for network in inv:
    for station in network:
        sta_start_dates.append([network.code+'.'+station.code, station.start_date])
sta_start_dates = pd.DataFrame(sta_start_dates, columns=['station','start_date'])

# Merge the start dates with the stations DataFrame
stations = pd.merge(stations, sta_start_dates, on='station', how='left')


### Plot with the boundaries for the velocity models

In [ ]:
# Load the data for the nodes of each region
df_pnsn_wa = pd.read_csv("../data/nodes_pnsn_wa.csv")
df_pnsn_jdf = pd.read_csv("../data/nodes_pnsn_jdf.csv")
df_pnsn_nor = pd.read_csv("../data/nodes_pnsn_nor.csv")
df_pnsn_sor = pd.read_csv("../data/nodes_pnsn_sor.csv")
df_swa_shore = pd.read_csv("../data/nodes_swa_shore.csv")    
df_swa_shelf_trench = pd.read_csv("../data/nodes_swa_shelf_trench.csv")
df_nwa_shore = pd.read_csv("../data/nodes_nwa_shore.csv")    
df_nwa_shelf_trench = pd.read_csv("../data/nodes_nwa_shelf_trench.csv")
df_or_shore = pd.read_csv("../data/nodes_or_shore.csv")
df_or_shelf_trench = pd.read_csv("../data/nodes_or_shelf_trench.csv")

In [ ]:


# Parameters for the plot
min_lat = 40
max_lat = 51
min_lon = -130
max_lon = -121

# Create a list to node sets
dfs = [df_pnsn_wa, df_pnsn_jdf, df_pnsn_nor, df_pnsn_sor, df_swa_shore, df_swa_shelf_trench,
       df_nwa_shore, df_nwa_shelf_trench, df_or_shore, df_or_shelf_trench]
colors = ['#1f78b4', '#a6cee3', '#e31a1c', '#33a02c', '#fb9a99',
          '#cab2d6', '#fdbf6f', '#ff7f00', '#b2df8a', '#6a3d9a']
region_names = ['PNSN WA', 'PNSN JdF', 'PNSN N. OR', 'PNSN S. OR', 'S. WA Shore', 'S. WA Shelf & Trench',
                       'N. WA Shore', 'N. WA Shelf & Trench', 'OR Shore', 'OR Shelf & Trench']

# Initialize the figure
fig = pygmt.Figure()

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Define target region
region = [min_lon, max_lon, min_lat, max_lat]
title = "All boundaries for velocity models"
fig.basemap(region=region, projection="M12c", frame=["af", f"WSne+t{title}"])

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")
pygmt.makecpt(cmap="viridis", series=[stations['active_days'].min(), stations['active_days'].max()])

# Define transformers
transformer_to_utm = Transformer.from_crs("epsg:4326", "epsg:32610", always_xy=True)  # WGS84 to UTM zone 10N
transformer_to_wgs84 = Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)  # UTM zone 10N to WGS84

for df, color,region in zip(dfs, colors,region_names):
    # Create a list to store the circles
    circles = []
    
    # Draw circles at the nodes
    for idx, row in df.iterrows():
        lon, lat = row["longitude"], row["latitude"]
        radius = (row['radius'] + row['buffer_zone']) * 1000  # Convert km to meters
        
        # Project to UTM
        x, y = transformer_to_utm.transform(lon, lat)
        
        # Create circle in UTM coordinates
        circle = Point(x, y).buffer(radius)
        
        # Project back to WGS84
        circle = transform(transformer_to_wgs84.transform, circle)
        
        circles.append(circle)
    
    # Create the union of all circles
    union_circles = unary_union(circles)
    
    # Convert the union to a GeoDataFrame
    gdf = gpd.GeoDataFrame(geometry=[union_circles])
    
       
    # Plot the union of circles
    for geom in gdf.geometry:
        if geom.geom_type == 'MultiPolygon':
            for poly in geom:
                x, y = poly.exterior.xy
                fig.plot(x=x, y=y, pen=f"1.8p,{color}", label=region)
        else:
            x, y = geom.exterior.xy
            fig.plot(x=x, y=y, pen=f"1.8,{color}", label=region)

fig.plot(
    x=stations["longitude"],
    y=stations["latitude"],
    style='t0.3c',
    cmap=True,
    pen=f"0.2p,black",
    fill=stations["active_days"],
    label="Stations",
)

# Plot a dummy point to add the station legend
fig.plot(x=[-110], y=[48], style="t0.3c", fill="white", label="Station", pen="0.2p,black")

# Add the legend to the plot
fig.legend(position="JTR+w4.4+jTR", box='+gwhite+p1p')
fig.colorbar(frame="xaf+lActive Days of Station")

# Save and display the figure
fig.savefig("../data/datasets_all_regions/nodes_all_w_stas_2010_2015_ver3.pdf")
fig.show()

### Plot without the boundaries for the velocity models

In [ ]:


# Parameters for the plot
min_lat = 38
max_lat = 51
min_lon = -130
max_lon = -121

# # Create a list to node sets
# dfs = [df_pnsn_wa, df_pnsn_jdf, df_pnsn_nor, df_pnsn_sor, df_swa_shore, df_swa_shelf_trench,
#        df_nwa_shore, df_nwa_shelf_trench, df_or_shore, df_or_shelf_trench]
# colors = ['#1f78b4', '#a6cee3', '#e31a1c', '#33a02c', '#fb9a99',
#           '#cab2d6', '#fdbf6f', '#ff7f00', '#b2df8a', '#6a3d9a']
# region_names = ['PNSN WA', 'PNSN JdF', 'PNSN N. OR', 'PNSN S. OR', 'S. WA Shore', 'S. WA Shelf & Trench',
#                        'N. WA Shore', 'N. WA Shelf & Trench', 'OR Shore', 'OR Shelf & Trench']

# Initialize the figure
fig = pygmt.Figure()

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Define target region
region = [min_lon, max_lon, min_lat, max_lat]
title = "All Stations with Picks"
fig.basemap(region=region, projection="M12c", frame=["af", f"WSne+t{title}"])

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")
pygmt.makecpt(cmap="viridis", series=[stations['active_days'].min(), stations['active_days'].max()])

# Define transformers
transformer_to_utm = Transformer.from_crs("epsg:4326", "epsg:32610", always_xy=True)  # WGS84 to UTM zone 10N
transformer_to_wgs84 = Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)  # UTM zone 10N to WGS84

# for df, color,region in zip(dfs, colors,region_names):
#     # Create a list to store the circles
#     circles = []
    
#     # Draw circles at the nodes
#     for idx, row in df.iterrows():
#         lon, lat = row["longitude"], row["latitude"]
#         radius = (row['radius'] + row['buffer_zone']) * 1000  # Convert km to meters
        
#         # Project to UTM
#         x, y = transformer_to_utm.transform(lon, lat)
        
#         # Create circle in UTM coordinates
#         circle = Point(x, y).buffer(radius)
        
#         # Project back to WGS84
#         circle = transform(transformer_to_wgs84.transform, circle)
        
#         circles.append(circle)
    
#     # Create the union of all circles
#     union_circles = unary_union(circles)
    
#     # Convert the union to a GeoDataFrame
#     gdf = gpd.GeoDataFrame(geometry=[union_circles])
    
       
#     # Plot the union of circles
#     for geom in gdf.geometry:
#         if geom.geom_type == 'MultiPolygon':
#             for poly in geom:
#                 x, y = poly.exterior.xy
#                 fig.plot(x=x, y=y, pen=f"1.8p,{color}", label=region)
#         else:
#             x, y = geom.exterior.xy
#             fig.plot(x=x, y=y, pen=f"1.8,{color}", label=region)

# Plot a dummy point to add the station legend
fig.plot(x=[-110], y=[48], style="t0.3c", fill="white", label="Station", pen="0.2p,black")

# Add plate boundaries
fig.plot(data="../data/datasets_all_regions/PB2002_boundaries.dig", pen="1.3p,dodgerblue4",transparency=50, label="Plate Boundaries")

# Plot the stations
fig.plot(
    x=stations["longitude"],
    y=stations["latitude"],
    style='t0.3c',
    cmap=True,
    pen=f"0.2p,black",
    fill=stations["active_days"],
    label="Stations",
)



# Add the legend to the plot
fig.legend(position="JTR+w3.6+jTR", box='+gwhite+p1p')
fig.colorbar(frame="xaf+lActive Days of Station")

# Save and display the figure
fig.savefig("../data/datasets_all_regions/nodes_all_w_stas_wo_boundaries_2010_2015_ver3.pdf")
fig.show()

### Plot without the plate boundaries

In [ ]:


# Parameters for the plot
min_lat = 39
max_lat = 51
min_lon = -130
max_lon = -121

# # Create a list to node sets
# dfs = [df_pnsn_wa, df_pnsn_jdf, df_pnsn_nor, df_pnsn_sor, df_swa_shore, df_swa_shelf_trench,
#        df_nwa_shore, df_nwa_shelf_trench, df_or_shore, df_or_shelf_trench]
# colors = ['#1f78b4', '#a6cee3', '#e31a1c', '#33a02c', '#fb9a99',
#           '#cab2d6', '#fdbf6f', '#ff7f00', '#b2df8a', '#6a3d9a']
# region_names = ['PNSN WA', 'PNSN JdF', 'PNSN N. OR', 'PNSN S. OR', 'S. WA Shore', 'S. WA Shelf & Trench',
#                        'N. WA Shore', 'N. WA Shelf & Trench', 'OR Shore', 'OR Shelf & Trench']

# Initialize the figure
fig = pygmt.Figure()

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Define target region
region = [min_lon, max_lon, min_lat, max_lat]
title = "(a)"
# fig.basemap(region=region, projection="M12c", frame=["af", f"WSne+t{title}+o0c/-1c"])
fig.basemap(region=region, projection="M12c", frame="af"
)
fig.text(
    x=-129.5, y=50.7, text=title,
    font="20p,Helvetica-Bold,black",
    no_clip=True    # This prevents the text from being clipped
)

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")
pygmt.makecpt(cmap="viridis", series=[stations['active_days'].min(), stations['active_days'].max()])

# Define transformers
transformer_to_utm = Transformer.from_crs("epsg:4326", "epsg:32610", always_xy=True)  # WGS84 to UTM zone 10N
transformer_to_wgs84 = Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)  # UTM zone 10N to WGS84

# for df, color,region in zip(dfs, colors,region_names):
#     # Create a list to store the circles
#     circles = []
    
#     # Draw circles at the nodes
#     for idx, row in df.iterrows():
#         lon, lat = row["longitude"], row["latitude"]
#         radius = (row['radius'] + row['buffer_zone']) * 1000  # Convert km to meters
        
#         # Project to UTM
#         x, y = transformer_to_utm.transform(lon, lat)
        
#         # Create circle in UTM coordinates
#         circle = Point(x, y).buffer(radius)
        
#         # Project back to WGS84
#         circle = transform(transformer_to_wgs84.transform, circle)
        
#         circles.append(circle)
    
#     # Create the union of all circles
#     union_circles = unary_union(circles)
    
#     # Convert the union to a GeoDataFrame
#     gdf = gpd.GeoDataFrame(geometry=[union_circles])
    
       
#     # Plot the union of circles
#     for geom in gdf.geometry:
#         if geom.geom_type == 'MultiPolygon':
#             for poly in geom:
#                 x, y = poly.exterior.xy
#                 fig.plot(x=x, y=y, pen=f"1.8p,{color}", label=region)
#         else:
#             x, y = geom.exterior.xy
#             fig.plot(x=x, y=y, pen=f"1.8,{color}", label=region)

# Plot a dummy point to add the station legend
fig.plot(x=[-110], y=[48], style="t0.3c", fill="white", label="Station", pen="0.2p,black")

# Add plate boundaries
# fig.plot(data="PB2002_boundaries.dig", pen="1.3p,dodgerblue4",transparency=50, label="Plate Boundaries")

# Plot the stations
fig.plot(
    x=stations["longitude"],
    y=stations["latitude"],
    style='t0.3c',
    cmap=True,
    pen=f"0.3p,black",
    fill=stations["active_days"],
    label="Stations",
)



# Add the legend to the plot
fig.legend(position="JTR+w2.0+jTR", box='+gwhite+p1p')
fig.colorbar(
    # Place the colorbar inside the plot (lowercase "j") in the Bottom Right (BR)
    # corner with an offset ("+o") of 0.7 centimeters and 0.3 centimeters in x or y
    # directions, respectively; move the x label above the horizontal colorbar ("+ml")
    position="jBC+o0.0c/0.25c+h+w8c/0.3c+ml",
    # Add a box around the colobar with a fill ("+g") in "white" color and a
    # transparency ("@") of 30 % and with a 0.8-points thick, black, outline ("+p")
    box="+gwhite@30+p0.8p,black",
    # Add x and y labels ("+l")
    frame=["x+lActive Days of Stations"],
)
# Save and display the figure
fig.savefig("../data/datasets_all_regions/stas_wo_boundaries_2010_2015_ver3.pdf")
# fig.savefig("../data/datasets_all_regions/stas_wo_boundaries_2010_2015_ver3.png", dpi=500)

fig.show()

## Make figure 1(b): events from ANSS and Morton-2023

### Load the ANSS catalog

In [ ]:
# Read Morton's catalog
events_morton = pd.read_csv('../data/ds01.csv')
# Convert the TSTRING to datetime
events_morton['datetime'] = pd.to_datetime(events_morton['TSTRING'], format='%Y%m%d%H%M%S', utc=True)
events_morton = events_morton.sort_values(by='DEPTH')
events_morton.head()

### Load the catalog from Morton-2023

In [ ]:
events_anss = pd.read_csv('../data/datasets_anss/anss_2010-15.csv',index_col=0)
events_anss['datetime'] = pd.to_datetime(events_anss['time'], format='%Y-%m-%dT%H:%M:%S.%fZ')
events_anss = events_anss.sort_values(by='depth')
events_anss.head()

### Map the events

### Plot with the boundaries for the velocity models

In [ ]:


# Parameters for the plot
min_lat = 40
max_lat = 51
min_lon = -130
max_lon = -121

# Create a list to node sets
dfs = [df_pnsn_wa, df_pnsn_jdf, df_pnsn_nor, df_pnsn_sor, df_swa_shore, df_swa_shelf_trench,
       df_nwa_shore, df_nwa_shelf_trench, df_or_shore, df_or_shelf_trench]
colors = ['#1f78b4', '#a6cee3', '#e31a1c', '#33a02c', '#fb9a99',
          '#cab2d6', '#fdbf6f', '#ff7f00', '#b2df8a', '#6a3d9a']
region_names = ['PNSN WA', 'PNSN JdF', 'PNSN N. OR', 'PNSN S. OR', 'S. WA Shore', 'S. WA Shelf & Trench',
                       'N. WA Shore', 'N. WA Shelf & Trench', 'OR Shore', 'OR Shelf & Trench']

# Initialize the figure

fig = pygmt.Figure()

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Define target region
region = [min_lon, max_lon, min_lat, max_lat]
title = "Events from Morton-2023 and ANSS"
fig.basemap(region=region, projection="M12c", frame=["af", f"WSne+t{title}"])

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# Define transformers
transformer_to_utm = Transformer.from_crs("epsg:4326", "epsg:32610", always_xy=True)  # WGS84 to UTM zone 10N
transformer_to_wgs84 = Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)  # UTM zone 10N to WGS84

for df, color,region in zip(dfs, colors,region_names):
    # Create a list to store the circles
    circles = []
    
    # Draw circles at the nodes
    for idx, row in df.iterrows():
        lon, lat = row["longitude"], row["latitude"]
        radius = (row['radius'] + row['buffer_zone']) * 1000  # Convert km to meters
        
        # Project to UTM
        x, y = transformer_to_utm.transform(lon, lat)
        
        # Create circle in UTM coordinates
        circle = Point(x, y).buffer(radius)
        
        # Project back to WGS84
        circle = transform(transformer_to_wgs84.transform, circle)
        
        circles.append(circle)
    
    # Create the union of all circles
    union_circles = unary_union(circles)
    
    # Convert the union to a GeoDataFrame
    gdf = gpd.GeoDataFrame(geometry=[union_circles])
    
       
    # Plot the union of circles
    for geom in gdf.geometry:
        if geom.geom_type == 'MultiPolygon':
            for poly in geom:
                x, y = poly.exterior.xy
                fig.plot(x=x, y=y, pen=f"1.8p,{color}", label=region)
        else:
            x, y = geom.exterior.xy
            fig.plot(x=x, y=y, pen=f"1.8,{color}", label=region)
fig.plot(
    x=events_anss["longitude"],
    y=events_anss["latitude"],
    style='c0.15c',
    pen=f"0.2p,black",
    fill='#d8b365',
    label="Morton-2023",
)

fig.plot(
    x=events_morton["LON"],
    y=events_morton["LAT"],
    style='c0.15c',
    pen=f"0.2p,black",
    fill='#5ab4ac',
    label="ANSS",
)



# Add the legend to the plot
fig.legend(position="JTR+w4.5+jTR", box='+gwhite+p1p')

# Save and display the figure
fig.savefig("../data/datasets_all_regions/fig1b.pdf")

fig.show()

### Plot without the boundaries for the velocity models

In [ ]:


# Parameters for the plot
min_lat = 39
max_lat = 51
min_lon = -130
max_lon = -121

# # Create a list to node sets
# dfs = [df_pnsn_wa, df_pnsn_jdf, df_pnsn_nor, df_pnsn_sor, df_swa_shore, df_swa_shelf_trench,
#        df_nwa_shore, df_nwa_shelf_trench, df_or_shore, df_or_shelf_trench]
# colors = ['#1f78b4', '#a6cee3', '#e31a1c', '#33a02c', '#fb9a99',
#           '#cab2d6', '#fdbf6f', '#ff7f00', '#b2df8a', '#6a3d9a']
# region_names = ['PNSN WA', 'PNSN JdF', 'PNSN N. OR', 'PNSN S. OR', 'S. WA Shore', 'S. WA Shelf & Trench',
#                        'N. WA Shore', 'N. WA Shelf & Trench', 'OR Shore', 'OR Shelf & Trench']

# Initialize the figure
fig = pygmt.Figure()

# Load grid data
grid = pygmt.datasets.load_earth_relief(resolution="15s", region=[min_lon, max_lon, min_lat, max_lat])
shade = pygmt.grdgradient(grid=grid, azimuth="0/90", normalize="t1")

# Define target region
region = [min_lon, max_lon, min_lat, max_lat]
title = "(b)"
fig.basemap(region=region, projection="M12c", frame="af")

fig.grdimage(grid=grid, cmap="blue", shading=shade, transparency=50)
fig.coast(region=region, projection="M12c", shorelines=True, borders=[1, 2], resolution="f")

# # Define transformers
# transformer_to_utm = Transformer.from_crs("epsg:4326", "epsg:32610", always_xy=True)  # WGS84 to UTM zone 10N
# transformer_to_wgs84 = Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)  # UTM zone 10N to WGS84

# for df, color,region in zip(dfs, colors,region_names):
#     # Create a list to store the circles
#     circles = []
    
#     # Draw circles at the nodes
#     for idx, row in df.iterrows():
#         lon, lat = row["longitude"], row["latitude"]
#         radius = (row['radius'] + row['buffer_zone']) * 1000  # Convert km to meters
        
#         # Project to UTM
#         x, y = transformer_to_utm.transform(lon, lat)
        
#         # Create circle in UTM coordinates
#         circle = Point(x, y).buffer(radius)
        
#         # Project back to WGS84
#         circle = transform(transformer_to_wgs84.transform, circle)
        
#         circles.append(circle)
    
#     # Create the union of all circles
#     union_circles = unary_union(circles)
    
#     # Convert the union to a GeoDataFrame
#     gdf = gpd.GeoDataFrame(geometry=[union_circles])
    
       
#     # Plot the union of circles
#     for geom in gdf.geometry:
#         if geom.geom_type == 'MultiPolygon':
#             for poly in geom:
#                 x, y = poly.exterior.xy
#                 fig.plot(x=x, y=y, pen=f"1.8p,{color}", label=region)
#         else:
#             x, y = geom.exterior.xy
#             fig.plot(x=x, y=y, pen=f"1.8,{color}", label=region)
fig.plot(
    x=events_anss["longitude"],
    y=events_anss["latitude"],
    style='c0.15c',
    pen=f"0.3p,black",
    fill='#d8b365',
    label="Morton-2023",
)

fig.plot(
    x=events_morton["LON"],
    y=events_morton["LAT"],
    style='c0.15c',
    pen=f"0.3p,black",
    fill='#5ab4ac',
    label="ANSS",
)


fig.text(
    x=-129.5, y=50.7, text=title,
    font="20p,Helvetica-Bold,black",
    no_clip=True    # This prevents the text from being clipped
)
# Add the legend to the plot
fig.legend(position="JTR+w2.6+jTR", box='+gwhite+p1p')

# Save and display the figure
fig.savefig("../data/datasets_all_regions/fig1b.pdf")
# fig.savefig("../data/datasets_all_regions/fig1b.png", dpi=500)

fig.show()

### Place the two maps in the same figure

In [ ]:

# Load the two existing PyGMT PDFs
doc1 = fitz.open("../data/datasets_all_regions/stas_wo_boundaries_2010_2015_ver3.pdf")
doc2 = fitz.open("../data/datasets_all_regions/fig1b.pdf")

# Create a new empty PDF to hold the combined plot
combined = fitz.open()

# Get dimensions of both plots
rect1 = doc1[0].rect
rect2 = doc2[0].rect

# Determine combined width and max height
combined_width = rect1.width + rect2.width
combined_height = max(rect1.height, rect2.height)

# Create a new blank page with combined size
page = combined.new_page(width=combined_width, height=combined_height)

# Paste the first plot on the left
page.show_pdf_page(rect1, doc1, 0)

# Paste the second plot to the right of the first
right_rect = fitz.Rect(rect1.width, 0, combined_width, rect2.height)
page.show_pdf_page(right_rect, doc2, 0)

# Save the combined PDF
combined.save("../data/datasets_all_regions/fig1.pdf")


In [ ]:
# Load the PDF
doc = fitz.open("../data/datasets_all_regions/fig1.pdf")

# Set zoom factor or resolution
zoom = 4  # 4x means ~300+ DPI depending on original figure size
mat = fitz.Matrix(zoom, zoom)

# Render the first page to a high-res pixmap
pix = doc[0].get_pixmap(matrix=mat, alpha=False)

# Save to PNG
pix.save("../data/datasets_all_regions/fig1.png")
